# Predict Students Dropout - Cleaning data

03_Predict_Students_Dropout_Cleanse_Data.ipynb

## Summary

This notebook is responsible for cleasing the raw data for the "Predict Students' Dropout and Academic Success" project. The data is read from row_data saved earlier in get data script.


## Steps

1. **Data Quality Check**
2. **Columns Rename**
3. **Check Duplicate Rows**
4. **Check Unique Values**
5. **Individual Variables Quality Check**


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np

In [ ]:

# Load the dataset into a Pandas dataframe
df_student = pd.read_csv('data/row_data.csv')

### Data Quality Check

In [ ]:
# Data Quality Check
def data_quality_check(df):
    print("Data Quality Check:\n")
    print("Shape of the DataFrame:", df.shape)
    print("Data Types:\n", df.dtypes)
    print("Missing Values:\n", df.isnull().sum())
    print("Summary Statistics:\n", df.describe(include='all'))

print("Data Quality Check for student_mat:")
data_quality_check(df_student)


#### Data Cleansing
The following code defines a function to cleanse and prepare the dataset for analysis. It includes steps for renaming columns, replacing spaces with underscores in column names, converting data types, and saving the cleaned data to a CSV file.

In [ ]:
def cleanse_data(df):
    # Rename specific columns
    df.rename(columns={
        'Nacionality': 'Nationality',
        "Mother's qualification": 'Mothers_qualification',
        "Father's qualification": 'Fathers_qualification',
        "Mother's occupation": 'Mothers_occupation',
        "Father's occupation": 'Fathers_occupation',
        'Daytime/evening attendance': 'Daytime_evening_attendance',
        'Curricular units 1st sem (credited)': 'Curricular_units_1st_sem_credited',
        'Curricular units 1st sem (enrolled)': 'Curricular_units_1st_sem_enrolled',
        'Curricular units 1st sem (evaluations)': 'Curricular_units_1st_sem_evaluations',
        'Curricular units 1st sem (approved)': 'Curricular_units_1st_sem_approved',
        'Curricular units 1st sem (grade)': 'Curricular_units_1st_sem_grade',
        'Curricular units 1st sem (without evaluations)': 'Curricular_units_1st_sem_without_evaluations',
        'Curricular units 2nd sem (credited)': 'Curricular_units_2nd_sem_credited',
        'Curricular units 2nd sem (enrolled)': 'Curricular_units_2nd_sem_enrolled',
        'Curricular units 2nd sem (evaluations)': 'Curricular_units_2nd_sem_evaluations',
        'Curricular units 2nd sem (approved)': 'Curricular_units_2nd_sem_approved',
        'Curricular units 2nd sem (grade)': 'Curricular_units_2nd_sem_grade',
        'Curricular units 2nd sem (without evaluations)': 'Curricular_units_2nd_sem_without_evaluations',
    }, inplace=True)
    
    # Replace spaces with underscores in all column names
    df.columns = df.columns.str.replace(' ', '_')
    
    
    # Convert data types if needed
    numeric_cols = [
        'Application_mode', 'Application_order', 'Admission_grade', 'Age_at_enrollment',
        'Curricular_units_1st_sem_credited', 'Curricular_units_1st_sem_enrolled', 
        'Curricular_units_1st_sem_evaluations', 'Curricular_units_1st_sem_approved', 
        'Curricular_units_1st_sem_grade', 'Curricular_units_1st_sem_without_evaluations',
        'Curricular_units_2nd_sem_credited', 'Curricular_units_2nd_sem_enrolled', 
        'Curricular_units_2nd_sem_evaluations', 'Curricular_units_2nd_sem_approved', 
        'Curricular_units_2nd_sem_grade', 'Curricular_units_2nd_sem_without_evaluations',
        'Unemployment_rate', 'Inflation_rate', 'GDP'
    ]
    
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

df_clean = cleanse_data(df_student)
display(df_clean.tail(5))

df_clean.columns

### Check Duplicate Rows

In [ ]:

# Display number of duplicated rows
print("Number of duplicated rows: ",df_student.duplicated().sum())

# Display rows with duplicates
print(df_student[df_student.duplicated()])

### Check Unique Values


In [ ]:

# Get no of Unique values in every column
for col in df_student.columns:
    if len(df_student[col].unique()) < 10:
        print(f"{col} has {len(df_student[col].unique())} unique values; Repeated values are {sorted(df_student[col].unique())}")
    else:
        print(f"{col} has {len(df_student[col].unique())} values")
     

### Individual Variables Quality Check

In [ ]:
# Zero columns:
def count_zeros(df):
    numerical_columns = df.select_dtypes(include = [np.number]).columns
    zeros_count = df[numerical_columns].eq(0).sum()
    return zeros_count

print(count_zeros(df_student))

# Dropping columns with zeroes:
def dropping(df, variable):
    df = df.drop(variable, axis = 1)
    return df

data = dropping(df_student, ["Curricular_units_1st_sem_credited", 
                            "Curricular_units_1st_sem_enrolled", 
                            "Curricular_units_1st_sem_without_evaluations", 
                            "Curricular_units_2nd_sem_without_evaluations"])



#### Treating Outliers in Numerical Columns
The following code identifies and treats outliers in the dataset using the Interquartile Range (IQR) method. It then saves the cleaned data to a CSV file.

In [ ]:
# List of numerical columns to check for outliers
numerical_columns = [
    'Previous_qualification_(grade)', 'Admission_grade', 
    'Age_at_enrollment', 'Curricular_units_1st_sem_credited',
    'Curricular_units_1st_sem_enrolled', 'Curricular_units_1st_sem_evaluations',
    'Curricular_units_1st_sem_approved', 'Curricular_units_1st_sem_grade',
    'Curricular_units_1st_sem_without_evaluations', 'Curricular_units_2nd_sem_credited',
    'Curricular_units_2nd_sem_enrolled', 'Curricular_units_2nd_sem_evaluations',
    'Curricular_units_2nd_sem_approved', 'Curricular_units_2nd_sem_grade',
    'Curricular_units_2nd_sem_without_evaluations', 'Unemployment_rate',
    'Inflation_rate', 'GDP'
]

# Function to detect and treat outliers using IQR method
def treat_outliers(df, columns):
    """
    Detects and treats outliers in the specified columns of the DataFrame using the IQR method.
    
    Args:
    df (pd.DataFrame): Input DataFrame to be treated.
    columns (list): List of numerical columns to check for outliers.
    
    Returns:
    pd.DataFrame: DataFrame with treated outliers.
    """
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Cap the outliers
        df[col] = df[col].apply(lambda x: lower_bound if x < lower_bound else x)
        df[col] = df[col].apply(lambda x: upper_bound if x > upper_bound else x)
        
        print(f'Outliers in {col} treated. Lower bound: {lower_bound}, Upper bound: {upper_bound}')
    
    return df

# Apply the function to treat outliers
df_cleaned = treat_outliers(df_student, numerical_columns)

# Save the cleaned data to a CSV file
df_cleaned.to_csv('data/cleaned_data.csv', index=False)